## Import 

In [1]:
import torch
import sys
import torch.nn as nn
import random  
import numpy as np
import matplotlib.pyplot as plt 
import pandas as pd
from matplotlib import __version__ as matplotlib_version


from sklearn import __version__ as sklearn_version
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split


## Version

In [2]:
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-learn version: {sklearn_version}")
print(f"Matplotlib: {matplotlib_version}")
print(f"Pandas: {pd.__version__}")

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version (from torch): {torch.version.cuda}")
    print(f"cuDNN version: {torch.backends.cudnn.version()}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

Python version: 3.13.2
PyTorch version: 2.8.0.dev20250409+cu128
NumPy version: 2.1.2
Scikit-learn version: 1.6.1
Matplotlib: 3.10.3
Pandas: 2.2.3
CUDA available: True
CUDA version (from torch): 12.8
cuDNN version: 90701
GPU device: NVIDIA GeForce RTX 3060 Laptop GPU


## Set seed 

In [3]:
def set_seed(seed=42):
    random.seed(seed)                        # Python random
    np.random.seed(seed)                     # NumPy random
    torch.manual_seed(seed)                  # PyTorch CPU
    torch.cuda.manual_seed(seed)             # PyTorch GPU
    torch.cuda.manual_seed_all(seed)         # PyTorch multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  

set_seed(42)


## Data preparetation

In [4]:
bc = load_breast_cancer()
X_numpy,y_numpy = bc.data , bc.target
print(X_numpy.shape)
print(y_numpy.shape)
print(X_numpy.dtype)
print(y_numpy.dtype)

(569, 30)
(569,)
float64
int64


## Data splitting

In [5]:
X_train,X_test,y_train,y_test = train_test_split(X_numpy,y_numpy,test_size=0.2,random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(455, 30)
(114, 30)
(455,)
(114,)


## Normalize

In [6]:
sc = RobustScaler() #output sklearn float64 defalue
X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

print(X_train.mean()) #original 
print(X_train_sc.mean()) #check normalized 

61.72624167942857
0.21110958695697554


## Numpy to Pytorch

In [7]:
X_train_sc_t = torch.tensor(X_train_sc,dtype=torch.float32)
X_test_sc_t = torch.tensor(X_test_sc,dtype=torch.float32)
y_train_t = torch.tensor(y_train,dtype=torch.float32).view(-1,1) #same size with train
y_test_t = torch.tensor(y_test,dtype=torch.float32).view(-1,1)

print(X_train_sc_t.size())
print(X_test_sc_t.size())
print(y_train_t.size())
print(y_test_t.size())

torch.Size([455, 30])
torch.Size([114, 30])
torch.Size([455, 1])
torch.Size([114, 1])


# Model 

In [8]:
class Logistic(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.linear = nn.Linear(input_size,1)

    def forward(self,X):
        y_predicted = torch.sigmoid(self.linear(X))
        return y_predicted
        

## Training

In [9]:
model = Logistic(X_train_sc_t.size()[1])
learning_rate = 0.1
criterion = nn.BCELoss()
optim = torch.optim.SGD(model.parameters(),lr=learning_rate)

In [10]:
n_epoch = 10
for epoch in range(n_epoch):
    #forward pass 
    y_predicted = model(X_train_sc_t)
    loss = criterion(y_predicted,y_train_t)

    #backward
    loss.backward()

    #update 
    optim.step()
    optim.zero_grad()

    print(f'epoch {epoch} | BCE {loss.item():.4}')

epoch 0 | BCE 0.8777
epoch 1 | BCE 0.6633
epoch 2 | BCE 0.5417
epoch 3 | BCE 0.4785
epoch 4 | BCE 0.4393
epoch 5 | BCE 0.4111
epoch 6 | BCE 0.3891
epoch 7 | BCE 0.3712
epoch 8 | BCE 0.3562
epoch 9 | BCE 0.3432


C:\Users\eieiz\AppData\Local\Temp\ipykernel_12324\272086872.py:14: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Scalar.cpp:23.)
  print(f'epoch {epoch} | BCE {loss.item():.4}')


## Test

In [20]:
with torch.no_grad():
    y_hat = model(X_test_sc_t)
    y_hat = y_hat.round()
    acc = y_hat.eq(y_test_t).sum().item() / float(y_test_t.size()[0])
    print(f"Accuracy: {acc:.4f}")

Accuracy: 0.9298
